# Exploratory Data Analysis — Iron Ore Drill Hole Database

**Objective:** Perform an initial exploration of the Vale Desenvolver drill hole database, understanding the structure, quality, and statistical distribution of geochemical variables (Fe, Si, G1, G2, G3) and lithological classification.

**Datasets:**
- `collar.csv` — Drill hole collar coordinates (365 holes)
- `assays.csv` — Geochemical assay intervals (5,487 samples)
- `block_model.csv` — Pre-existing block model grid (2,594 blocks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = os.path.join("..", "data")

## 1. Loading the Datasets

In [ ]:
collar = pd.read_csv(os.path.join(DATA_DIR, "collar.csv"))
assays = pd.read_csv(os.path.join(DATA_DIR, "assays.csv"))
block_model = pd.read_csv(os.path.join(DATA_DIR, "block_model.csv"))

print(f"Collar:      {collar.shape[0]:>6,} rows × {collar.shape[1]} columns")
print(f"Assays:      {assays.shape[0]:>6,} rows × {assays.shape[1]} columns")
print(f"Block Model: {block_model.shape[0]:>6,} rows × {block_model.shape[1]} columns")

## 2. Collar Table — Drill Hole Locations

In [ ]:
collar.head(10)

In [ ]:
collar.info()
print("\n")
collar.describe()

### 2.1 Coordinate Consistency Check

The collar table uses UTM coordinates. We expect X values around 641,000 and Y values around 8,426,000–8,428,000. Let's verify this — some rows may have swapped X and Y.

In [ ]:
# Detect rows where X and Y appear to be swapped
# Normal range: X ~ 641,000 | Y ~ 8,425,000–8,429,000
swapped_mask = collar["X"] > 1_000_000  # Y values accidentally in the X column

print(f"Rows with swapped coordinates: {swapped_mask.sum()} out of {len(collar)}")
collar[swapped_mask].head()

### 2.2 Drill Hole Location Map

In [ ]:
# Fix swapped coordinates for plotting
collar_plot = collar.copy()
mask = collar_plot["X"] > 1_000_000
collar_plot.loc[mask, ["X", "Y"]] = collar_plot.loc[mask, ["Y", "X"]].values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plan view (X vs Y)
axes[0].scatter(collar_plot["X"], collar_plot["Y"], s=15, c="steelblue", edgecolors="k", linewidths=0.3)
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")
axes[0].set_title("Drill Hole Collar Locations — Plan View")
axes[0].ticklabel_format(style="plain")

# Depth distribution
axes[1].hist(collar_plot["PROF"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Total Depth (m)")
axes[1].set_ylabel("Number of Drill Holes")
axes[1].set_title("Distribution of Drill Hole Depths")

plt.tight_layout()
plt.show()

print(f"Depth range: {collar_plot['PROF'].min():.1f} m – {collar_plot['PROF'].max():.1f} m")
print(f"Mean depth:  {collar_plot['PROF'].mean():.1f} m")

## 3. Assay Table — Geochemical Data

In [ ]:
assays.head(10)

In [ ]:
assays.info()
print("\n")
assays.describe()

### 3.1 Missing Values Analysis

The value **-99** is used as a sentinel for missing data in the geochemical columns (FE, SI, G1, G2, G3). Let's quantify this.

In [ ]:
geochem_cols = ["FE", "SI", "G1", "G2", "G3"]

# Count sentinel values (-99) per column
sentinel_counts = (assays[geochem_cols] == -99).sum()
sentinel_pct = (sentinel_counts / len(assays) * 100).round(1)

missing_summary = pd.DataFrame({
    "Sentinel (-99) Count": sentinel_counts,
    "Percentage (%)": sentinel_pct,
    "Valid Samples": len(assays) - sentinel_counts
})
missing_summary

### 3.2 Distribution of Geochemical Variables

Histograms for Fe and Si (excluding -99 sentinels) to understand the grade distribution.

In [ ]:
# Replace sentinel values with NaN for clean analysis
assays_clean = assays.copy()
for col in geochem_cols:
    assays_clean[col] = assays_clean[col].replace(-99, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fe distribution
fe_data = assays_clean["FE"].dropna()
axes[0].hist(fe_data, bins=40, color="#c0392b", edgecolor="white", alpha=0.85)
axes[0].axvline(fe_data.mean(), color="black", linestyle="--", linewidth=1.5, label=f"Mean = {fe_data.mean():.1f}%")
axes[0].set_xlabel("Fe (%)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Iron (Fe) Grade Distribution")
axes[0].legend()

# Si distribution
si_data = assays_clean["SI"].dropna()
axes[1].hist(si_data, bins=40, color="#2980b9", edgecolor="white", alpha=0.85)
axes[1].axvline(si_data.mean(), color="black", linestyle="--", linewidth=1.5, label=f"Mean = {si_data.mean():.1f}%")
axes[1].set_xlabel("SiO₂ (%)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Silica (SiO₂) Grade Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Granulometry Variables (G1, G2, G3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

granulometry = {"G1 (Coarse)": "G1", "G2 (Medium)": "G2", "G3 (Fine)": "G3"}
colors = ["#27ae60", "#f39c12", "#8e44ad"]

for ax, (label, col), color in zip(axes, granulometry.items(), colors):
    data = assays_clean[col].dropna()
    if len(data) > 0:
        ax.hist(data, bins=30, color=color, edgecolor="white", alpha=0.85)
        ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.5, label=f"Mean = {data.mean():.1f}%")
        ax.set_xlabel(f"{label} (%)")
        ax.set_ylabel("Frequency")
        ax.set_title(f"{label} Distribution")
        ax.legend()
    else:
        ax.set_title(f"{label} — No valid data")

plt.tight_layout()
plt.show()

### 3.4 Fe vs SiO₂ Scatter Plot

In iron ore deposits, Fe and Si typically show a strong inverse relationship — higher iron content correlates with lower silica.

In [ ]:
valid = assays_clean.dropna(subset=["FE", "SI"])

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(valid["FE"], valid["SI"], c=valid["FE"], cmap="RdYlGn", s=8, alpha=0.6, edgecolors="none")
ax.set_xlabel("Fe (%)", fontsize=12)
ax.set_ylabel("SiO₂ (%)", fontsize=12)
ax.set_title("Fe vs SiO₂ — Inverse Correlation", fontsize=14)
plt.colorbar(scatter, ax=ax, label="Fe (%)")
plt.tight_layout()
plt.show()

corr = valid["FE"].corr(valid["SI"])
print(f"Pearson correlation (Fe × Si): {corr:.3f}")

### 3.5 Correlation Matrix

In [ ]:
corr_matrix = assays_clean[geochem_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=1, ax=ax,
            vmin=-1, vmax=1)
ax.set_title("Correlation Matrix — Geochemical Variables", fontsize=13)
plt.tight_layout()
plt.show()

### 3.6 Lithotype Distribution

Frequency of each lithological type in the assay database.

In [ ]:
lito_counts = assays["Lito_Final"].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(lito_counts.index, lito_counts.values, color="steelblue", edgecolor="white")
ax.set_xlabel("Lithotype")
ax.set_ylabel("Sample Count")
ax.set_title("Lithotype Distribution in Assay Database")
ax.bar_label(bars, fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 3.7 Boxplot — Fe by Lithotype

In [ ]:
# Order lithotypes by median Fe
lito_order = assays_clean.groupby("Lito_Final")["FE"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=assays_clean, x="Lito_Final", y="FE", order=lito_order, ax=ax,
            palette="RdYlGn", fliersize=2)
ax.axhline(y=56, color="red", linestyle="--", linewidth=1, label="Cutoff Fe = 56%")
ax.set_xlabel("Lithotype")
ax.set_ylabel("Fe (%)")
ax.set_title("Fe Grade Distribution by Lithotype")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Block Model Overview

In [ ]:
block_model.head(10)

In [ ]:
block_model.info()
print("\n")
block_model.describe()

In [ ]:
# Block model spatial extent
print("Block Model Spatial Extent:")
print(f"  X: {block_model['X'].min():,.0f} – {block_model['X'].max():,.0f} m")
print(f"  Y: {block_model['Y'].min():,.0f} – {block_model['Y'].max():,.0f} m")
print(f"  Z: {block_model['Z'].min():,.0f} – {block_model['Z'].max():,.0f} m")
print(f"\nLithotype distribution in block model:")
print(block_model["LITOTIPO"].value_counts())

## 5. Key Findings

1. **365 drill holes** covering a dense grid pattern in the Desenvolver deposit area.
2. **Coordinate issues detected** — some collar rows have X and Y columns swapped (values > 1M in X column). This will be corrected in the data treatment phase.
3. **Sentinel values (-99)** are used for missing geochemical data, especially prevalent in G1, G2, G3 granulometry columns.
4. **Fe and Si show strong inverse correlation**, consistent with typical iron ore geology.
5. **Lithotype variety** — multiple ore types (HF, HEM, CM, etc.) and waste types are present.
6. **Block model** contains 2,594 pre-defined blocks with existing (but mostly sentinel) grade values to be re-estimated.

**Next step:** Data treatment — clean sentinel values, fix coordinates, and prepare the dataset for block model estimation.